In [3]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# ==========================================
# LOAD MODEL-READY DATA
# ==========================================

df = pd.read_csv(
    "../data/processed/transactions_model_ready.csv",
    low_memory=False
)

df["timestamp"] = pd.to_datetime(df["timestamp"])

print("Dataset:", df.shape)

Dataset: (999891, 26)


In [4]:
# ==========================================
# TIME-BASED SPLIT
# ==========================================

train_mask = df["timestamp"] < "2026-08-01"

val_mask = (
    (df["timestamp"] >= "2026-08-01") &
    (df["timestamp"] < "2026-09-01")
)

test_mask = df["timestamp"] >= "2026-09-01"

train_df = df.loc[train_mask].copy()
val_df = df.loc[val_mask].copy()
test_df = df.loc[test_mask].copy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (776632, 26)
Validation: (113448, 26)
Test: (109811, 26)


In [5]:
# ==========================================
# CREATE FEATURES AND TARGET
# ==========================================

target = "is_fraud"

drop_columns = [
    "transaction_id",
    "id",
    "user_id",
    "merchant_id",
    "device_id",
    "ip_address",
    "timestamp",
    "previous_timestamp",

    # Full-dataset exploratory features
    "device_user_count",
    "device_transaction_count",
    "location_transaction_count",
    "user_location_transaction_count",
    "user_merchant_transaction_count",
    "user_category_transaction_count",
    "new_merchant_for_user"
]

feature_columns = [
    col for col in df.columns
    if col not in drop_columns + [target]
]

X_train = train_df[feature_columns]
y_train = train_df[target]

X_val = val_df[feature_columns]
y_val = val_df[target]

X_test = test_df[feature_columns]
y_test = test_df[target]

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (776632, 19)
X_val: (113448, 19)
X_test: (109811, 19)


In [6]:
print("\nFraud distribution:")

for name, y in [
    ("TRAIN", y_train),
    ("VALIDATION", y_val),
    ("TEST", y_test)
]:
    print(f"\n{name}")
    print(y.value_counts())
    print(f"Fraud rate: {y.mean() * 100:.2f}%")


Fraud distribution:

TRAIN
is_fraud
0.0    543644
1.0    232988
Name: count, dtype: int64
Fraud rate: 30.00%

VALIDATION
is_fraud
0.0    79347
1.0    34101
Name: count, dtype: int64
Fraud rate: 30.06%

TEST
is_fraud
0.0    76918
1.0    32893
Name: count, dtype: int64
Fraud rate: 29.95%


In [7]:
categorical_cols = [
    "merchant_category",
    "location",
    "payment_method"
]

numeric_cols = [
    col for col in X_train.columns
    if col not in categorical_cols
]

In [8]:
categorical_cols = [
    "merchant_category",
    "location",
    "payment_method"
]

numeric_cols = [
    col for col in X_train.columns
    if col not in categorical_cols
]

print("Numerical features:", len(numeric_cols))
print("Categorical features:", len(categorical_cols))

print("\nNumerical columns:")
print(numeric_cols)

Numerical features: 16
Categorical features: 3

Numerical columns:
['amount', 'account_age', 'transaction_count_24h', 'avg_transaction_amount', 'distance_from_home', 'is_new_device', 'is_new_location', 'flag_missing_amount', 'flag_negative_amount', 'flag_missing_user', 'flag_missing_merchant', 'flag_negative_distance', 'flag_duplicate_transaction', 'flag_invalid_timestamp', 'flag_unknown_category', 'data_quality_issue']


In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Numerical preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=True
    )),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="most_frequent"
    )),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

# Combine pipelines
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])

print("Preprocessor created successfully.")

Preprocessor created successfully.


In [10]:
X_train_processed = preprocessor.fit_transform(X_train)

X_val_processed = preprocessor.transform(X_val)

X_test_processed = preprocessor.transform(X_test)

print("X_train processed:", X_train_processed.shape)
print("X_val processed:", X_val_processed.shape)
print("X_test processed:", X_test_processed.shape)

X_train processed: (776632, 46)
X_val processed: (113448, 46)
X_test processed: (109811, 46)


**Logistic Regression**

In [11]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

print("Training Logistic Regression...")

logistic_model.fit(
    X_train_processed,
    y_train
)

print("Training completed.")

Training Logistic Regression...
Training completed.


In [12]:
y_val_pred = logistic_model.predict(X_val_processed)

y_val_prob = logistic_model.predict_proba(
    X_val_processed
)[:, 1]

print("Validation predictions generated.")

Validation predictions generated.


In [14]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

accuracy = accuracy_score(y_val, y_val_pred)*100
precision = precision_score(y_val, y_val_pred)*100
recall = recall_score(y_val, y_val_pred)*100
f1 = f1_score(y_val, y_val_pred)*100
roc_auc = roc_auc_score(y_val, y_val_prob)*100
pr_auc = average_precision_score(y_val, y_val_prob)*100

print("===== LOGISTIC REGRESSION — VALIDATION =====")

print(f"Accuracy : {accuracy:.2f}%")
print(f"Precision: {precision:.2f}%")
print(f"Recall   : {recall:.2f}%")
print(f"F1 Score : {f1:.2f}%")
print(f"ROC-AUC  : {roc_auc:.2f}%")
print(f"PR-AUC   : {pr_auc:.2f}%")

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred))

print("\nClassification Report:")
print(classification_report(y_val, y_val_pred))

===== LOGISTIC REGRESSION — VALIDATION =====
Accuracy : 93.35%
Precision: 90.01%
Recall   : 87.61%
F1 Score : 88.79%
ROC-AUC  : 97.11%
PR-AUC   : 95.76%

Confusion Matrix:
[[76033  3314]
 [ 4226 29875]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.95      0.96      0.95     79347
         1.0       0.90      0.88      0.89     34101

    accuracy                           0.93    113448
   macro avg       0.92      0.92      0.92    113448
weighted avg       0.93      0.93      0.93    113448



**Random Forest**

In [ ]:
# ==========================================
# RANDOM FOREST
# ==========================================

from sklearn.ensemble import RandomForestClassifier

random_forest_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("Training Random Forest...")

random_forest_model.fit(
    X_train_processed,
    y_train
    
)

print("Training completed.")

Training Random Forest...
Training completed.


In [16]:
# ==========================================
# RANDOM FOREST VALIDATION
# ==========================================

rf_val_pred = random_forest_model.predict(
    X_val_processed
)

rf_val_prob = random_forest_model.predict_proba(
    X_val_processed
)[:, 1]

print("Predictions generated.")

Predictions generated.


In [17]:
# ==========================================
# RANDOM FOREST METRICS
# ==========================================

rf_accuracy = accuracy_score(y_val, rf_val_pred)*100
rf_precision = precision_score(y_val, rf_val_pred)*100
rf_recall = recall_score(y_val, rf_val_pred)*100
rf_f1 = f1_score(y_val, rf_val_pred)*100
rf_roc_auc = roc_auc_score(y_val, rf_val_prob)*100
rf_pr_auc = average_precision_score(y_val, rf_val_prob)*100

print("===== RANDOM FOREST — VALIDATION =====")

print(f"Accuracy : {rf_accuracy:.2f}%")
print(f"Precision: {rf_precision:.2f}%")
print(f"Recall   : {rf_recall:.2f}%")
print(f"F1 Score : {rf_f1:.2f}%")
print(f"ROC-AUC  : {rf_roc_auc:.2f}%")
print(f"PR-AUC   : {rf_pr_auc:.2f}%")

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, rf_val_pred))

print("\nClassification Report:")
print(classification_report(y_val, rf_val_pred))

===== RANDOM FOREST — VALIDATION =====
Accuracy : 95.20%
Precision: 97.28%
Recall   : 86.45%
F1 Score : 91.54%
ROC-AUC  : 97.69%
PR-AUC   : 96.42%

Confusion Matrix:
[[78522   825]
 [ 4622 29479]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.94      0.99      0.97     79347
         1.0       0.97      0.86      0.92     34101

    accuracy                           0.95    113448
   macro avg       0.96      0.93      0.94    113448
weighted avg       0.95      0.95      0.95    113448



**XGBoost**

In [19]:
# ==========================================
# XGBOOST
# ==========================================

from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

print("Training XGBoost...")

xgb_model.fit(
    X_train_processed,
    y_train
)

print("Training completed.")

Training XGBoost...
Training completed.


In [20]:
# ==========================================
# XGBOOST VALIDATION
# ==========================================

xgb_val_pred = xgb_model.predict(
    X_val_processed
)

xgb_val_prob = xgb_model.predict_proba(
    X_val_processed
)[:, 1]

print("Predictions generated.")

Predictions generated.


In [22]:
# ==========================================
# XGBOOST METRICS
# ==========================================

xgb_accuracy = accuracy_score(y_val, xgb_val_pred)*100
xgb_precision = precision_score(y_val, xgb_val_pred)*100
xgb_recall = recall_score(y_val, xgb_val_pred)*100
xgb_f1 = f1_score(y_val, xgb_val_pred)*100
xgb_roc_auc = roc_auc_score(y_val, xgb_val_prob)*100
xgb_pr_auc = average_precision_score(y_val, xgb_val_prob)*100

print("===== XGBOOST — VALIDATION =====")

print(f"Accuracy : {xgb_accuracy:.2f}%")
print(f"Precision: {xgb_precision:.2f}%")
print(f"Recall   : {xgb_recall:.2f}%")
print(f"F1 Score : {xgb_f1:.2f}%")
print(f"ROC-AUC  : {xgb_roc_auc:.2f}%")
print(f"PR-AUC   : {xgb_pr_auc:.2f}%")

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, xgb_val_pred))

print("\nClassification Report:")
print(classification_report(y_val, xgb_val_pred))

===== XGBOOST — VALIDATION =====
Accuracy : 96.21%
Precision: 99.05%
Recall   : 88.24%
F1 Score : 93.33%
ROC-AUC  : 98.14%
PR-AUC   : 97.02%

Confusion Matrix:
[[79057   290]
 [ 4010 30091]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.95      1.00      0.97     79347
         1.0       0.99      0.88      0.93     34101

    accuracy                           0.96    113448
   macro avg       0.97      0.94      0.95    113448
weighted avg       0.96      0.96      0.96    113448



In [23]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(
    y_val,
    xgb_val_prob
)

# F1 for each threshold
f1_scores = (
    2 * precision[:-1] * recall[:-1]
    / (precision[:-1] + recall[:-1] + 1e-10)
)

best_idx = np.argmax(f1_scores)

best_threshold = thresholds[best_idx]
best_f1 = f1_scores[best_idx]

print(f"Best F1 threshold: {best_threshold:.4f}")
print(f"Best F1 score: {best_f1:.4f}")

Best F1 threshold: 0.4654
Best F1 score: 0.9334


In [24]:
from sklearn.metrics import classification_report, confusion_matrix

optimal_val_pred = (
    xgb_val_prob >= best_threshold
).astype(int)

print("===== OPTIMAL F1 THRESHOLD =====")

print(confusion_matrix(
    y_val,
    optimal_val_pred
))

print("\nClassification Report:")
print(classification_report(
    y_val,
    optimal_val_pred
))

===== OPTIMAL F1 THRESHOLD =====
[[79021   326]
 [ 3971 30130]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.95      1.00      0.97     79347
         1.0       0.99      0.88      0.93     34101

    accuracy                           0.96    113448
   macro avg       0.97      0.94      0.95    113448
weighted avg       0.96      0.96      0.96    113448



In [25]:
# ==========================================
# FINAL TEST EVALUATION
# ==========================================

test_pred = (
    xgb_model.predict_proba(X_test_processed)[:, 1]
    >= best_threshold
).astype(int)

test_prob = xgb_model.predict_proba(
    X_test_processed
)[:, 1]

In [26]:
# ==========================================
# FINAL TEST METRICS
# ==========================================

test_accuracy = accuracy_score(y_test, test_pred)*100
test_precision = precision_score(y_test, test_pred)*100
test_recall = recall_score(y_test, test_pred)*100
test_f1 = f1_score(y_test, test_pred)*100
test_roc_auc = roc_auc_score(y_test, test_prob)*100
test_pr_auc = average_precision_score(y_test, test_prob)*100

print("===== FINGUARD AI — FINAL TEST =====")

print(f"Accuracy : {test_accuracy:.2f}%")
print(f"Precision: {test_precision:.2f}%")
print(f"Recall   : {test_recall:.2f}%")
print(f"F1 Score : {test_f1:.2f}%")
print(f"ROC-AUC  : {test_roc_auc:.2f}%")
print(f"PR-AUC   : {test_pr_auc:.2f}%")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_pred))

print("\nClassification Report:")
print(classification_report(y_test, test_pred))

===== FINGUARD AI — FINAL TEST =====
Accuracy : 96.13%
Precision: 98.89%
Recall   : 88.07%
F1 Score : 93.17%
ROC-AUC  : 98.14%
PR-AUC   : 96.98%

Confusion Matrix:
[[76593   325]
 [ 3923 28970]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.95      1.00      0.97     76918
         1.0       0.99      0.88      0.93     32893

    accuracy                           0.96    109811
   macro avg       0.97      0.94      0.95    109811
weighted avg       0.96      0.96      0.96    109811



In [27]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(
    xgb_model,
    "../models/xgboost_model.pkl"
)

joblib.dump(
    preprocessor,
    "../models/preprocessor.pkl"
)

print("Model and preprocessor saved successfully.")

Model and preprocessor saved successfully.


In [28]:
model_config = {
    "model": "XGBoost",
    "threshold": float(best_threshold),
    "validation_f1": float(best_f1),
    "test_f1": float(test_f1),
    "test_precision": float(test_precision),
    "test_recall": float(test_recall),
    "test_roc_auc": float(test_roc_auc),
    "test_pr_auc": float(test_pr_auc)
}

joblib.dump(
    model_config,
    "../models/model_config.pkl"
)

print(model_config)

{'model': 'XGBoost', 'threshold': 0.4653724730014801, 'validation_f1': 0.9334386665548653, 'test_f1': 93.16910014793851, 'test_precision': 98.89059566478922, 'test_recall': 88.07345027817469, 'test_roc_auc': 98.13953434756384, 'test_pr_auc': 96.98330453462674}
